# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors.

## PyTorch setup

In [181]:
import torch

In [182]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [183]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [184]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [185]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [186]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [187]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [188]:
import torch.nn as nn

In [ ]:
input_size = train_data[0][0].numel()

# I think to(device) is not needed here but I added it just in case
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

## Defining the optimizer

In [190]:
import torch.optim as optim

In [191]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Defining loss function

In [192]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [193]:
import numpy as np

In [194]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [195]:
from tqdm import tqdm

In [196]:
def train(model, train_data, optimizer, criterion, epochs, batch_size):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)

        # Eval
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    return epoch_train_loss, epoch_val_loss

In [197]:
loss = train(model, train_data, optimizer, criterion, 30, 64)

Epoch 1/30: 100%|██████████| 782/782 [00:04<00:00, 188.21it/s]


Epoch [1/30], Loss: 0.3913, Val Loss: 0.2065


Epoch 2/30: 100%|██████████| 782/782 [00:03<00:00, 200.92it/s]


Epoch [2/30], Loss: 0.1925, Val Loss: 0.1442


Epoch 3/30: 100%|██████████| 782/782 [00:03<00:00, 200.99it/s]


...

Epoch [28/30], Loss: 0.0096, Val Loss: 0.1328


Epoch 29/30: 100%|██████████| 782/782 [00:02<00:00, 267.02it/s]


Epoch [29/30], Loss: 0.0026, Val Loss: 0.1227


Epoch 30/30: 100%|██████████| 782/782 [00:03<00:00, 256.30it/s]


Epoch [30/30], Loss: 0.0040, Val Loss: 0.1342


In [198]:
print(f"Final Training loss: {loss[0]:.4f}, Final Validation loss: {loss[1]:.4f}")

Final Training loss: 0.0040, Final Validation loss: 0.1342
